# Pandas

**Objectif :** Maîtriser pandas comme brique de base avant ML/DL, avec réflexes rapides utilisables en compétition (temps limité, données propres/sales, feature engineering express).

## Comment utiliser ce notebook
- Chaque section a : **Cours** → **Mémo-tech** → **Exemple** → **Exercice** → **Solution** (cachée, essaie d'abord seul·e).
- Exécute chaque cellule dans l'ordre, une fois lue.
- Le rythme conseillé est indicatif adapte-le au niveau des candidats IOAI que tu coaches.

## Plan des 3 s

|  | Thème | Objectif |
|---|---|---|
| **1** | Fondamentaux | Series, DataFrame, sélection, filtrage, valeurs manquantes, stats de base |
| **2** | Intermédiaire | GroupBy, merge/join/concat, pivot, apply/map, strings, dates |
| **3** | Optimisation & pont vers le ML/DL | Vectorisation, dtypes, MultiIndex, method chaining, feature engineering, mini-projet |

---


In [1]:
# Setup général à exécuter en premier
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)


pandas : 3.0.2
numpy  : 2.4.4


#  1  Fondamentaux

## 1.1  Series : le brique de base

Une **Series** est un tableau 1D **étiqueté** (un index + des valeurs). C'est la colonne élémentaire de pandas.

> **Mémo-tech : "Series = dico avec ordre + super-pouvoirs numpy"**
> Pense à une Series comme un dictionnaire (`clé → valeur`) mais qui garde l'ordre et sait faire du calcul vectorisé.


In [2]:
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(s)
print("\nValeur à l'index 'b' :", s['b'])
print("Type :", type(s))
print("Valeurs (numpy array) :", s.values)
print("Index :", s.index.tolist())


a    10
b    20
c    30
d    40
dtype: int64

Valeur à l'index 'b' : 20
Type : <class 'pandas.Series'>
Valeurs (numpy array) : [10 20 30 40]
Index : ['a', 'b', 'c', 'd']


## 1.2 DataFrame : le tableau 2D

Un **DataFrame** = plusieurs Series alignées sur le même index = un tableau (lignes × colonnes), comme une feuille Excel.

> **Mémo-tech : "DataFrame = dict de colonnes"**
> `pd.DataFrame({"col1": [...], "col2": [...]})` chaque clé devient une colonne.


In [3]:
df = pd.DataFrame({
    "nom": ["Aïcha", "Boureima", "Chantal", "Drissa"],
    "age": [22, 25, 21, 24],
    "score_ml": [78.5, 91.2, 65.0, 88.8],
    "pays": ["BF", "BF", "SN", "CI"]
})
df


,nom,age,score_ml,pays
0,Aïcha,22,78.5,BF
1,Boureima,25,91.2,BF
2,Chantal,21,65.0,SN
3,Drissa,24,88.8,CI


In [4]:
print("Dimensions (lignes, colonnes) :", df.shape)
print("\nTypes de colonnes :")
print(df.dtypes)
print("\nInfos générales :")
df.info()


Dimensions (lignes, colonnes) : (4, 4)

Types de colonnes :
nom             str
age           int64
score_ml    float64
pays            str
dtype: object

Infos générales :
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   nom       4 non-null      str    
 1   age       4 non-null      int64  
 2   score_ml  4 non-null      float64
 3   pays      4 non-null      str    
dtypes: float64(1), int64(1), str(2)
memory usage: 260.0 bytes


## 1.3 Lecture de fichiers

En compétition ou en projet réel, les données arrivent presque toujours via un fichier.

> **Mémo-tech : "read_X pour lire, to_X pour écrire"** symétrie totale.
> `pd.read_csv()` ↔ `df.to_csv()`, `pd.read_excel()` ↔ `df.to_excel()`, `pd.read_json()` ↔ `df.to_json()`.


In [5]:
# On simule un CSV en mémoire pour l'exemple (pas besoin de fichier réel)
from io import StringIO

csv_data = StringIO('''nom,age,score_ml,pays
Aicha,22,78.5,BF
Boureima,25,91.2,BF
Chantal,21,65.0,SN
Drissa,24,88.8,CI''')

df2 = pd.read_csv(csv_data)
df2


,nom,age,score_ml,pays
0,Aicha,22,78.5,BF
1,Boureima,25,91.2,BF
2,Chantal,21,65.0,SN
3,Drissa,24,88.8,CI


## 1.4 Sélection : `loc` vs `iloc`

C'est LE piège classique des débutants.

> **Mémo-tech : "LOC = Label OrienteC, ILOC = Integer LOCation"**
> - `.loc[label]` → tu donnes des **noms** (étiquettes d'index / colonnes)
> - `.iloc[position]` → tu donnes des **positions entières** (comme des indices de liste Python, base 0)
> - Astuce : **L**oc = **L**abel, **I**loc = **I**ndex entier.


In [6]:
df_idx = df.set_index("nom")
print(df_idx)

print("\n--- loc : par label ---")
print(df_idx.loc["Boureima"])          # une ligne, par nom
print(df_idx.loc["Boureima", "age"])   # une cellule précise

print("\n--- iloc : par position ---")
print(df_idx.iloc[1])          # 2e ligne (position 1)
print(df_idx.iloc[1, 0])       # 2e ligne, 1re colonne
print(df_idx.iloc[0:2, 0:2])   # slicing façon numpy


          age  score_ml pays
nom                         
Aïcha      22      78.5   BF
Boureima   25      91.2   BF
Chantal    21      65.0   SN
Drissa     24      88.8   CI

--- loc : par label ---
age           25
score_ml    91.2
pays          BF
Name: Boureima, dtype: object
25

--- iloc : par position ---
age           25
score_ml    91.2
pays          BF
Name: Boureima, dtype: object
25
          age  score_ml
nom                    
Aïcha      22      78.5
Boureima   25      91.2


**Piège fréquent** : avec `.loc`, le slicing `df.loc[0:2]` **inclut** la borne de fin.
Avec `.iloc`, `df.iloc[0:2]` **exclut** la borne de fin (comme les slices Python classiques).


## 1.5 Filtrage (masques booléens)

> **Mémo-tech : "Un masque booléen = une passoire"**
> `df[condition]` ne garde que les lignes où `condition` vaut `True`. Combine avec `&` (et), `|` (ou), `~` (non)  **jamais** `and`/`or`/`not` sur des Series !


In [7]:
# Une seule condition
print(df[df["score_ml"] > 80])

print()
# Conditions combinées : parenthèses obligatoires autour de chaque condition
print(df[(df["score_ml"] > 70) & (df["pays"] == "BF")])

print()
# .isin() pour tester l'appartenance à une liste
print(df[df["pays"].isin(["BF", "CI"])])


        nom  age  score_ml pays
1  Boureima   25      91.2   BF
3    Drissa   24      88.8   CI

        nom  age  score_ml pays
0     Aïcha   22      78.5   BF
1  Boureima   25      91.2   BF

        nom  age  score_ml pays
0     Aïcha   22      78.5   BF
1  Boureima   25      91.2   BF
3    Drissa   24      88.8   CI


## 1.6 Valeurs manquantes (NaN)

> **Mémo-tech : "ISNA cherche les trous, DROPNA les jette, FILLNA les bouche"**


In [8]:
df_nan = df.copy()
df_nan.loc[1, "score_ml"] = np.nan
df_nan.loc[3, "pays"] = np.nan
print(df_nan)

print("\nOù sont les NaN ?")
print(df_nan.isna())

print("\nCombien de NaN par colonne ?")
print(df_nan.isna().sum())

print("\nSupprimer les lignes avec au moins un NaN :")
print(df_nan.dropna())

print("\nRemplir les NaN (score -> moyenne, pays -> 'Inconnu') :")
df_filled = df_nan.copy()
df_filled["score_ml"] = df_filled["score_ml"].fillna(df_filled["score_ml"].mean())
df_filled["pays"] = df_filled["pays"].fillna("Inconnu")
print(df_filled)


        nom  age  score_ml pays
0     Aïcha   22      78.5   BF
1  Boureima   25       NaN   BF
2   Chantal   21      65.0   SN
3    Drissa   24      88.8  NaN

Où sont les NaN ?
     nom    age  score_ml   pays
0  False  False     False  False
1  False  False      True  False
2  False  False     False  False
3  False  False     False   True

Combien de NaN par colonne ?
nom         0
age         0
score_ml    1
pays        1
dtype: int64

Supprimer les lignes avec au moins un NaN :
       nom  age  score_ml pays
0    Aïcha   22      78.5   BF
2  Chantal   21      65.0   SN

Remplir les NaN (score -> moyenne, pays -> 'Inconnu') :
        nom  age   score_ml     pays
0     Aïcha   22  78.500000       BF
1  Boureima   25  77.433333       BF
2   Chantal   21  65.000000       SN
3    Drissa   24  88.800000  Inconnu


## 1.7 Statistiques descriptives de base

> **Mémo-tech : "DESCRIBE = le bulletin de notes du DataFrame"**


In [9]:
print(df.describe())          # résumé statistique des colonnes numériques
print()
print("Moyenne score_ml :", df["score_ml"].mean())
print("Médiane :", df["score_ml"].median())
print("Écart-type :", df["score_ml"].std())
print("Min / Max :", df["score_ml"].min(), "/", df["score_ml"].max())
print("Nb de valeurs par pays :")
print(df["pays"].value_counts())


             age   score_ml
count   4.000000   4.000000
mean   23.000000  80.875000
std     1.825742  11.931296
min    21.000000  65.000000
25%    21.750000  75.125000
50%    23.000000  83.650000
75%    24.250000  89.400000
max    25.000000  91.200000

Moyenne score_ml : 80.875
Médiane : 83.65
Écart-type : 11.931296381086732
Min / Max : 65.0 / 91.2
Nb de valeurs par pays :
pays
BF    2
SN    1
CI    1
Name: count, dtype: int64


---
## Cheat-sheet  1

| Opération | Code |
|---|---|
| Créer une Series | `pd.Series([...], index=[...])` |
| Créer un DataFrame | `pd.DataFrame({"col": [...]})` |
| Lire un CSV | `pd.read_csv("fichier.csv")` |
| Dimensions | `df.shape` |
| Types de colonnes | `df.dtypes` / `df.info()` |
| Sélection par label | `df.loc[label, col]` |
| Sélection par position | `df.iloc[i, j]` |
| Filtrer | `df[condition]`, `df[(c1) & (c2)]` |
| Appartenance | `df[col].isin([...])` |
| NaN : détecter | `df.isna()`, `df.isna().sum()` |
| NaN : supprimer | `df.dropna()` |
| NaN : remplir | `df.fillna(valeur)` |
| Résumé stats | `df.describe()` |
| Comptage catégories | `df[col].value_counts()` |

---
## Exercices  1


In [10]:
# Jeu de données pour les exercices de la  1
data_ex1 = pd.DataFrame({
    "candidat": ["A", "B", "C", "D", "E", "F"],
    "pays": ["BF", "BF", "SN", "CI", "BF", "SN"],
    "age": [19, 21, np.nan, 23, 20, 22],
    "score_qcm": [72, 88, 91, np.nan, 65, 78],
    "score_prog": [80, 95, 70, 60, np.nan, 85],
})
data_ex1


,candidat,pays,age,score_qcm,score_prog
0,A,BF,19.0,72.0,80.0
1,B,BF,21.0,88.0,95.0
2,C,SN,NaN,91.0,70.0
3,D,CI,23.0,NaN,60.0
4,E,BF,20.0,65.0,NaN
5,F,SN,22.0,78.0,85.0


**Exercice 1.1** Affiche uniquement les candidats du Burkina Faso (BF) avec `score_qcm > 70`.

**Exercice 1.2** Combien de valeurs manquantes y a-t-il au total dans `data_ex1` ?

**Exercice 1.3** Remplace les NaN de `age` par la médiane, et les NaN de `score_qcm`/`score_prog` par 0.

**Exercice 1.4** Calcule un score final = `(score_qcm + score_prog) / 2`, ajoute-le comme nouvelle colonne, et affiche le candidat avec le score final le plus élevé (utilise `.loc` + `.idxmax()`).

Essaie dans la cellule ci-dessous avant de regarder la solution !


In [11]:
# Ton espace pour essayer les exercices 1.1 à 1.4





### Solutions (déplie seulement après avoir essayé)

In [12]:
# --- 1.1 ---
sol_1_1 = data_ex1[(data_ex1["pays"] == "BF") & (data_ex1["score_qcm"] > 70)]
print("1.1 :\n", sol_1_1, "\n")

# --- 1.2 ---
sol_1_2 = data_ex1.isna().sum().sum()
print("1.2 : total NaN =", sol_1_2, "\n")

# --- 1.3 ---
df_1_3 = data_ex1.copy()
df_1_3["age"] = df_1_3["age"].fillna(df_1_3["age"].median())
df_1_3["score_qcm"] = df_1_3["score_qcm"].fillna(0)
df_1_3["score_prog"] = df_1_3["score_prog"].fillna(0)
print("1.3 :\n", df_1_3, "\n")

# --- 1.4 ---
df_1_4 = df_1_3.copy()
df_1_4["score_final"] = (df_1_4["score_qcm"] + df_1_4["score_prog"]) / 2
meilleur = df_1_4.loc[df_1_4["score_final"].idxmax()]
print("1.4 :\n", df_1_4, "\n")
print("Meilleur candidat :\n", meilleur)


1.1 :
   candidat pays   age  score_qcm  score_prog
0        A   BF  19.0       72.0        80.0
1        B   BF  21.0       88.0        95.0 

1.2 : total NaN = 3 

1.3 :
   candidat pays   age  score_qcm  score_prog
0        A   BF  19.0       72.0        80.0
1        B   BF  21.0       88.0        95.0
2        C   SN  21.0       91.0        70.0
3        D   CI  23.0        0.0        60.0
4        E   BF  20.0       65.0         0.0
5        F   SN  22.0       78.0        85.0 

1.4 :
   candidat pays   age  score_qcm  score_prog  score_final
0        A   BF  19.0       72.0        80.0         76.0
1        B   BF  21.0       88.0        95.0         91.5
2        C   SN  21.0       91.0        70.0         80.5
3        D   CI  23.0        0.0        60.0         30.0
4        E   BF  20.0       65.0         0.0         32.5
5        F   SN  22.0       78.0        85.0         81.5 

Meilleur candidat :
 candidat          B
pays             BF
age            21.0
score_qcm     

---
#  2 Intermédiaire

## 2.1 GroupBy : diviser, appliquer, combiner

> **Mémo-tech : "GroupBy = Split → Apply → Combine" (SAC)**
> On **découpe** (split) les données par groupe, on **applique** une fonction (apply), puis on **recombine** (combine) le résultat.


In [13]:
df_g = pd.DataFrame({
    "pays": ["BF", "BF", "SN", "CI", "BF", "SN"],
    "domaine": ["ML", "DL", "ML", "DL", "ML", "DL"],
    "score": [72, 88, 91, 60, 65, 78],
})

print(df_g.groupby("pays")["score"].mean())
print()
print(df_g.groupby(["pays", "domaine"])["score"].mean())
print()
# .agg() pour plusieurs statistiques d'un coup
print(df_g.groupby("pays")["score"].agg(["mean", "min", "max", "count"]))


pays
BF    75.0
CI    60.0
SN    84.5
Name: score, dtype: float64

pays  domaine
BF    DL         88.0
      ML         68.5
CI    DL         60.0
SN    DL         78.0
      ML         91.0
Name: score, dtype: float64

      mean  min  max  count
pays                       
BF    75.0   65   88      3
CI    60.0   60   60      1
SN    84.5   78   91      2


## 2.2 Merge, Join, Concat

> **Mémo-tech : "MERGE = comme un JOIN SQL, CONCAT = coller des blocs"**
> - `pd.merge()` : combine deux DataFrames selon une **clé commune** (comme `JOIN` en SQL) `how="inner"/"left"/"right"/"outer"`.
> - `pd.concat()` : empile des DataFrames **verticalement** (lignes) ou **horizontalement** (colonnes), sans logique de clé.


In [14]:
candidats = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "nom": ["Aicha", "Boureima", "Chantal", "Drissa"],
})

resultats = pd.DataFrame({
    "id": [1, 2, 3, 5],
    "score": [78, 91, 65, 50],
})

print("inner (intersection des id) :")
print(pd.merge(candidats, resultats, on="id", how="inner"))

print("\nleft (tous les candidats, même sans score) :")
print(pd.merge(candidats, resultats, on="id", how="left"))

print("\nouter (tout le monde, NaN si absent d'un côté) :")
print(pd.merge(candidats, resultats, on="id", how="outer"))

print("\nconcat vertical (empiler deux DataFrames) :")
print(pd.concat([candidats, candidats], ignore_index=True))


inner (intersection des id) :
   id       nom  score
0   1     Aicha     78
1   2  Boureima     91
2   3   Chantal     65

left (tous les candidats, même sans score) :
   id       nom  score
0   1     Aicha   78.0
1   2  Boureima   91.0
2   3   Chantal   65.0
3   4    Drissa    NaN

outer (tout le monde, NaN si absent d'un côté) :
   id       nom  score
0   1     Aicha   78.0
1   2  Boureima   91.0
2   3   Chantal   65.0
3   4    Drissa    NaN
4   5       NaN   50.0

concat vertical (empiler deux DataFrames) :
   id       nom
0   1     Aicha
1   2  Boureima
2   3   Chantal
3   4    Drissa
4   1     Aicha
5   2  Boureima
6   3   Chantal
7   4    Drissa


## 2.3 Tableaux croisés : `pivot_table` et `crosstab`

> **Mémo-tech : "PIVOT_TABLE = Excel TCD (tableau croisé dynamique)"**


In [15]:
df_piv = pd.DataFrame({
    "pays": ["BF", "BF", "SN", "SN", "CI", "CI"],
    "domaine": ["ML", "DL", "ML", "DL", "ML", "DL"],
    "score": [72, 88, 91, 60, 65, 78],
})

piv = df_piv.pivot_table(index="pays", columns="domaine", values="score", aggfunc="mean")
print(piv)

print()
print(pd.crosstab(df_piv["pays"], df_piv["domaine"]))  # compte les occurrences


domaine    DL    ML
pays               
BF       88.0  72.0
CI       78.0  65.0
SN       60.0  91.0

domaine  DL  ML
pays           
BF        1   1
CI        1   1
SN        1   1


## 2.4 `apply`, `map`, `applymap` (→ `.map` sur DataFrame depuis pandas 2.1)

> **Mémo-tech : "MAP = 1 colonne (Series), APPLY = ligne/colonne, MAP DataFrame = toute la grille"**
> - `Series.map(fonction_ou_dict)` : transforme chaque élément d'**une** colonne.
> - `DataFrame.apply(fonction, axis=...)` : applique une fonction à chaque **ligne** (`axis=1`) ou **colonne** (`axis=0`).
> - `DataFrame.map(fonction)` : applique élément par élément sur **toute la grille** (remplace `applymap`, déprécié).


In [16]:
df_a = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

# map sur une Series (transformation élément par élément)
print(df_a["a"].map(lambda x: x ** 2))

# apply sur colonnes (axis=0, par défaut) : une fonction résumant chaque colonne
print(df_a.apply(lambda col: col.max() - col.min()))

# apply sur lignes (axis=1) : une fonction combinant les colonnes d'une même ligne
print(df_a.apply(lambda row: row["a"] + row["b"], axis=1))

# map sur tout le DataFrame (élément par élément, grille entière)
print(df_a.map(lambda x: x + 1))


0    1
1    4
2    9
Name: a, dtype: int64
a     2
b    20
dtype: int64
0    11
1    22
2    33
dtype: int64
   a   b
0  2  11
1  3  21
2  4  31


**Règle d'or de performance** : `apply`/`map` sont pratiques mais **lents** (boucle Python cachée).
Dès que possible, préfère les opérations **vectorisées** natives (`df["a"] * 2`, `np.where(...)`, etc.) voir  3.


## 2.5 Méthodes `.str` pour le texte

> **Mémo-tech : "Toujours passer par `.str` pour manipuler du texte dans une Series"**


In [17]:
noms = pd.Series(["Aicha KABORE", "boureima Sawadogo", "CHANTAL diallo"])

print(noms.str.lower())
print(noms.str.title())
print(noms.str.contains("sawadogo", case=False))
print(noms.str.split(" ").str[0])   # premier mot (prénom)
print(noms.str.len())               # longueur de chaque chaîne


0         aicha kabore
1    boureima sawadogo
2       chantal diallo
dtype: str
0         Aicha Kabore
1    Boureima Sawadogo
2       Chantal Diallo
dtype: str
0    False
1     True
2    False
dtype: bool
0       Aicha
1    boureima
2     CHANTAL
dtype: object
0    12
1    17
2    14
dtype: int64


## 2.6 Dates et séries temporelles

> **Mémo-tech : "to_datetime pour convertir, .dt pour tout extraire"**


In [18]:
dates = pd.to_datetime(["2026-01-15", "2026-03-02", "2026-07-07"])
s_dates = pd.Series(dates)

print(s_dates.dt.year)
print(s_dates.dt.month)
print(s_dates.dt.day_name())

# Créer une plage de dates (utile pour des séries temporelles synthétiques)
rng = pd.date_range("2026-01-01", periods=5, freq="D")
print(rng)


0    2026
1    2026
2    2026
dtype: int32
0    1
1    3
2    7
dtype: int32
0    Thursday
1      Monday
2     Tuesday
dtype: str
DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05'], dtype='datetime64[us]', freq='D')


## 2.7 Tri et rang

> **Mémo-tech : "sort_values trie les données, sort_index trie l'étiquette, rank donne le classement"**


In [19]:
df_sort = pd.DataFrame({"nom": ["A", "B", "C", "D"], "score": [78, 91, 65, 88]})

print(df_sort.sort_values("score", ascending=False))
print()
print(df_sort.assign(rang=df_sort["score"].rank(ascending=False)))


  nom  score
1   B     91
3   D     88
0   A     78
2   C     65



  nom  score  rang
0   A     78   3.0
1   B     91   1.0
2   C     65   4.0
3   D     88   2.0


---
## Cheat-sheet

| Opération | Code |
|---|---|
| Grouper + agréger | `df.groupby(col)["x"].mean()` / `.agg([...])` |
| Fusion type SQL | `pd.merge(df1, df2, on="cle", how="inner/left/right/outer")` |
| Empiler | `pd.concat([df1, df2], ignore_index=True)` |
| Tableau croisé | `df.pivot_table(index=, columns=, values=, aggfunc=)` |
| Comptage croisé | `pd.crosstab(s1, s2)` |
| Transformer 1 colonne | `Series.map(fn_ou_dict)` |
| Fonction ligne/colonne | `df.apply(fn, axis=0/1)` |
| Fonction élément par élément (grille) | `df.map(fn)` |
| Texte | `s.str.lower()/.contains()/.split()/.len()` |
| Dates | `pd.to_datetime(s)`, `s.dt.year/.month/.day_name()` |
| Tri | `df.sort_values(col)`, `s.rank()` |

---
##  Exercices


In [20]:
# Jeu de données pour les exercices de la  2
participants = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "nom": ["Fatou Traore", "Issa Ouedraogo", "Awa Kone", "Karim Diallo", "Rokia Sy", "Yacouba Bah"],
    "pays": ["BF", "BF", "CI", "SN", "CI", "BF"],
})

notes = pd.DataFrame({
    "id": [1, 2, 3, 4, 6, 7],
    "matiere": ["ML", "ML", "DL", "ML", "DL", "ML"],
    "note": [82, 74, 91, 68, 85, 60],
    "date": pd.to_datetime(["2026-02-01", "2026-02-01", "2026-02-03",
                             "2026-02-02", "2026-02-04", "2026-02-05"]),
})

print(participants)
print()
print(notes)


   id             nom pays
0   1    Fatou Traore   BF
1   2  Issa Ouedraogo   BF
2   3        Awa Kone   CI
3   4    Karim Diallo   SN
4   5        Rokia Sy   CI
5   6     Yacouba Bah   BF

   id matiere  note       date
0   1      ML    82 2026-02-01
1   2      ML    74 2026-02-01
2   3      DL    91 2026-02-03
3   4      ML    68 2026-02-02
4   6      DL    85 2026-02-04
5   7      ML    60 2026-02-05


**Exercice 2.1** Fusionne `participants` et `notes` avec un `left join` sur `id` (garde tous les participants).

**Exercice 2.2** Calcule la note moyenne par pays.

**Exercice 2.3** Crée une colonne `initiale_prenom` à partir de `nom` (première lettre du premier mot, en majuscule) via `.str`.

**Exercice 2.4** Extrais le mois (`.dt.month`) de chaque date dans `notes`, et fais un `pivot_table` note moyenne par (matière x mois).


In [21]:
# Ton espace pour essayer les exercices 2.1 à 2.4





### Solutions

In [22]:
# --- 2.1 ---
fusion = pd.merge(participants, notes, on="id", how="left")
print("2.1 :\n", fusion, "\n")

# --- 2.2 ---
moy_pays = fusion.groupby("pays")["note"].mean()
print("2.2 :\n", moy_pays, "\n")

# --- 2.3 ---
participants_2_3 = participants.copy()
participants_2_3["initiale_prenom"] = participants_2_3["nom"].str.split(" ").str[0].str[0].str.upper()
print("2.3 :\n", participants_2_3, "\n")

# --- 2.4 ---
notes_2_4 = notes.copy()
notes_2_4["mois"] = notes_2_4["date"].dt.month
piv_2_4 = notes_2_4.pivot_table(index="matiere", columns="mois", values="note", aggfunc="mean")
print("2.4 :\n", piv_2_4)


2.1 :
    id             nom pays matiere  note       date
0   1    Fatou Traore   BF      ML  82.0 2026-02-01
1   2  Issa Ouedraogo   BF      ML  74.0 2026-02-01
2   3        Awa Kone   CI      DL  91.0 2026-02-03
3   4    Karim Diallo   SN      ML  68.0 2026-02-02
4   5        Rokia Sy   CI     NaN   NaN        NaT
5   6     Yacouba Bah   BF      DL  85.0 2026-02-04 

2.2 :
 pays
BF    80.333333
CI    91.000000
SN    68.000000
Name: note, dtype: float64 

2.3 :
    id             nom pays initiale_prenom
0   1    Fatou Traore   BF               F
1   2  Issa Ouedraogo   BF               I
2   3        Awa Kone   CI               A
3   4    Karim Diallo   SN               K
4   5        Rokia Sy   CI               R
5   6     Yacouba Bah   BF               Y 



2.4 :
 mois        2
matiere      
DL       88.0
ML       71.0


---
#  3 Optimisation & pont vers le ML/DL

## 3.1 Vectorisation vs `apply` vs boucle `for` : le match de performance

> **Mémo-tech : "Boucle for = à pied, apply = à vélo, vectorisation = en voiture"**
> En compétition, le temps compte : la vectorisation numpy/pandas peut être **10 à 100x plus rapide** qu'une boucle Python.


In [23]:
import time

n = 200_000
df_perf = pd.DataFrame({"x": np.random.randn(n)})

# 1) Boucle for (à éviter absolument)
t0 = time.time()
resultat_boucle = []
for val in df_perf["x"][:20_000]:   # réduit à 20k pour ne pas attendre trop longtemps
    resultat_boucle.append(val ** 2 + 1)
t_boucle = time.time() - t0

# 2) apply
t0 = time.time()
resultat_apply = df_perf["x"][:20_000].apply(lambda x: x ** 2 + 1)
t_apply = time.time() - t0

# 3) Vectorisé (numpy/pandas natif)
t0 = time.time()
resultat_vect = df_perf["x"][:20_000] ** 2 + 1
t_vect = time.time() - t0

print(f"Boucle for : {t_boucle*1000:.2f} ms")
print(f"apply      : {t_apply*1000:.2f} ms")
print(f"Vectorisé  : {t_vect*1000:.2f} ms")


Boucle for : 4.52 ms
apply      : 15.18 ms
Vectorisé  : 0.40 ms


>  **Astuce compétition** : dès que tu écris une boucle `for` sur un DataFrame/Series, demande-toi
> "est-ce qu'il existe une opération vectorisée (`+`, `-`, `np.where`, `.clip`, `.isin`...) qui fait la même chose ?"


## 3.2 `np.where` et `np.select` : des `if/else` vectorisés

> **Mémo-tech : "np.where = if/else vectorisé, np.select = switch/case vectorisé"**


In [24]:
df_w = pd.DataFrame({"score": [45, 72, 88, 60, 95]})

# if/else vectorisé
df_w["mention"] = np.where(df_w["score"] >= 70, "Admis", "Non admis")

# switch/case vectorisé (plusieurs conditions)
conditions = [
    df_w["score"] >= 90,
    df_w["score"] >= 70,
    df_w["score"] >= 50,
]
choix = ["Excellent", "Bien", "Passable"]
df_w["niveau"] = np.select(conditions, choix, default="Insuffisant")

print(df_w)


   score    mention       niveau
0     45  Non admis  Insuffisant
1     72      Admis         Bien
2     88      Admis         Bien
3     60  Non admis     Passable
4     95      Admis    Excellent


## 3.3 Optimisation mémoire : les `dtypes`

> **Mémo-tech : "category pour le texte répétitif, downcast pour les nombres"**
> Sur de gros datasets (fréquent en ML), le choix du type change radicalement la mémoire utilisée.


In [25]:
df_mem = pd.DataFrame({
    "pays": np.random.choice(["BF", "SN", "CI", "TG", "ML"], size=100_000),
    "valeur": np.random.randint(0, 1000, size=100_000),
})

avant = df_mem.memory_usage(deep=True).sum() / 1024**2
df_mem["pays"] = df_mem["pays"].astype("category")
df_mem["valeur"] = pd.to_numeric(df_mem["valeur"], downcast="integer")
apres = df_mem.memory_usage(deep=True).sum() / 1024**2

print(f"Mémoire avant optimisation : {avant:.2f} Mo")
print(f"Mémoire après optimisation : {apres:.2f} Mo")
print(f"Gain : {(1 - apres/avant)*100:.1f} %")


Mémoire avant optimisation : 5.63 Mo
Mémoire après optimisation : 0.29 Mo
Gain : 94.9 %


## 3.4 `query()` : filtrage plus lisible (et parfois plus rapide)

> **Mémo-tech : "query() = écrire le filtre comme une phrase"**


In [26]:
df_q = pd.DataFrame({"pays": ["BF", "SN", "CI"], "score": [78, 91, 65]})

# Équivalent : df_q[(df_q["score"] > 70) & (df_q["pays"] == "SN")]
print(df_q.query("score > 70 and pays == 'SN'"))


  pays  score
1   SN     91


## 3.5 MultiIndex : hiérarchiser les données

> **Mémo-tech : "MultiIndex = un index à plusieurs étages (comme des dossiers imbriqués)"**


In [27]:
df_multi = pd.DataFrame({
    "pays": ["BF", "BF", "SN", "SN"],
    "annee": [2025, 2026, 2025, 2026],
    "score": [70, 78, 85, 91],
}).set_index(["pays", "annee"])

print(df_multi)
print()
print("Accès à un pays :")
print(df_multi.loc["BF"])
print()
print("Accès à une combinaison précise :")
print(df_multi.loc[("SN", 2026)])


            score
pays annee       
BF   2025      70
     2026      78
SN   2025      85
     2026      91

Accès à un pays :
       score
annee       
2025      70
2026      78

Accès à une combinaison précise :
score    91
Name: (SN, 2026), dtype: int64


## 3.6 Method chaining avec `.pipe()`

> **Mémo-tech : "pipe() = chaîner les étapes comme une pipeline de traitement (Unix `|`)"**
> Très utile pour un pipeline de nettoyage/feature engineering lisible et réutilisable.


In [28]:
def nettoyer(df):
    return df.dropna()

def normaliser_score(df, colonne="score"):
    df = df.copy()
    df[colonne] = (df[colonne] - df[colonne].mean()) / df[colonne].std()
    return df

def ajouter_mention(df, colonne="score"):
    df = df.copy()
    df["mention"] = np.where(df[colonne] > 0, "Au-dessus de la moyenne", "En dessous")
    return df

df_pipe = pd.DataFrame({"score": [78, 91, 65, np.nan, 88]})

resultat = (
    df_pipe
    .pipe(nettoyer)
    .pipe(normaliser_score, colonne="score")
    .pipe(ajouter_mention, colonne="score")
)
print(resultat)


      score                  mention
0 -0.213072               En dessous
1  0.894901  Au-dessus de la moyenne
2 -1.321044               En dessous
4  0.639215  Au-dessus de la moyenne


## 3.7 Pont vers le ML/DL : de pandas à numpy/tensors

> **Mémo-tech : "DataFrame → .values/.to_numpy() → array numpy → tensor"**
> C'est l'étape que tu feras à chaque projet : nettoyer avec pandas, puis convertir pour scikit-learn / PyTorch / TensorFlow.


In [29]:
df_ml = pd.DataFrame({
    "age": [22, 25, 21, 24, 19],
    "score_qcm": [72, 88, 65, 91, 60],
    "score_prog": [80, 95, 70, 85, 55],
    "admis": [1, 1, 0, 1, 0],   # variable cible (target)
})

# Séparer features (X) et target (y)
X = df_ml.drop(columns=["admis"]).to_numpy()   # -> array numpy, prêt pour sklearn/torch
y = df_ml["admis"].to_numpy()

print("X (features) :\n", X)
print("\ny (target) :", y)
print("\nType de X :", type(X), "| shape :", X.shape)


X (features) :
 [[22 72 80]
 [25 88 95]
 [21 65 70]
 [24 91 85]
 [19 60 55]]

y (target) : [1 1 0 1 0]

Type de X : <class 'numpy.ndarray'> | shape : (5, 3)


In [30]:
# One-hot encoding avec pandas (utile avant un modèle ML classique)
df_cat = pd.DataFrame({"pays": ["BF", "SN", "CI", "BF"]})
print(pd.get_dummies(df_cat, columns=["pays"], dtype=int))


   pays_BF  pays_CI  pays_SN
0        1        0        0
1        0        0        1
2        0        1        0
3        1        0        0


>  En pratique pour l'IOAI : `pandas` sert à **explorer/nettoyer/feature-engineer**, puis tu passes le relais
> à `numpy`/`scikit-learn`/`torch`/`tensorflow` pour l'entraînement. Sais reconnaître ce moment de bascule : dès que tu
> as une matrice numérique propre (`X`) et une cible (`y`), pandas a fini son travail.


---
## Cheat-sheet  3 (optimisation)

| Besoin | Solution rapide |
|---|---|
| Éviter les boucles `for` | Opérations vectorisées (`+`, `-`, `.clip()`, `np.where`) |
| if/else vectorisé | `np.where(condition, si_vrai, si_faux)` |
| switch/case vectorisé | `np.select([cond1, cond2, ...], [choix1, choix2, ...], default=...)` |
| Réduire la mémoire (texte répétitif) | `df[col].astype("category")` |
| Réduire la mémoire (nombres) | `pd.to_numeric(df[col], downcast="integer"/"float")` |
| Filtrage lisible | `df.query("condition")` |
| Index hiérarchique | `df.set_index([col1, col2])` |
| Chaîner des étapes de nettoyage | `df.pipe(fonction1).pipe(fonction2)` |
| Passer au ML | `df.to_numpy()`, `pd.get_dummies()` |

---
##  Exercices  3 (mini-projet de synthèse)


In [31]:
# Dataset de synthèse pour le mini-projet final
np.random.seed(42)
n = 500
mini_projet = pd.DataFrame({
    "candidat_id": range(1, n + 1),
    "pays": np.random.choice(["BF", "SN", "CI", "TG", "NE"], size=n),
    "age": np.random.randint(17, 26, size=n),
    "score_qcm": np.round(np.random.normal(70, 15, size=n), 1),
    "score_prog": np.round(np.random.normal(65, 20, size=n), 1),
    "date_inscription": pd.date_range("2026-01-01", periods=n, freq="h"),
})

# On introduit volontairement des valeurs manquantes et aberrantes (réalisme)
mini_projet.loc[np.random.choice(n, 30, replace=False), "score_qcm"] = np.nan
mini_projet.loc[np.random.choice(n, 20, replace=False), "score_prog"] = np.nan
mini_projet.loc[np.random.choice(n, 5, replace=False), "score_qcm"] = 999  # aberrant

mini_projet.head()


,candidat_id,pays,age,score_qcm,score_prog,date_inscription
0,1,TG,20,59.2,57.7,2026-01-01 00:00:00
1,2,NE,25,83.6,80.1,2026-01-01 01:00:00
2,3,CI,22,59.9,58.6,2026-01-01 02:00:00
3,4,NE,19,60.2,84.0,2026-01-01 03:00:00
4,5,NE,17,76.7,84.6,2026-01-01 04:00:00


**Mini-projet à faire de bout en bout :**

1. Repère et corrige les valeurs aberrantes (`score_qcm == 999` → remplace par NaN, puis par la médiane).
2. Remplis les NaN restants de `score_prog` par la médiane du **pays** correspondant (indice : `groupby().transform()`).
3. Crée une colonne `score_final = 0.5 * score_qcm + 0.5 * score_prog`.
4. Crée une colonne `mois_inscription` à partir de `date_inscription`.
5. Fais un `pivot_table` : score final moyen par (pays x mois_inscription).
6. Optimise les dtypes (`pays` en category, entiers en downcast).
7. Prépare `X` (features numériques : age, score_qcm, score_prog) et `y` (crée une cible `admis = 1 si score_final > 70 sinon 0`), sous forme de tableaux numpy prêts pour scikit-learn.

C'est exactement ce type de pipeline (nettoyage → feature engineering → export numpy) que tu retrouveras dans un vrai projet ML/DL, y compris en conditions de compétition.


In [32]:
# Ton espace pour le mini-projet (étapes 1 à 7)





### Solution complète Mini-projet

In [33]:
df_proj = mini_projet.copy()

# 1) Valeurs aberrantes -> NaN -> médiane
df_proj["score_qcm"] = df_proj["score_qcm"].replace(999, np.nan)
df_proj["score_qcm"] = df_proj["score_qcm"].fillna(df_proj["score_qcm"].median())

# 2) NaN de score_prog -> médiane par pays (transform garde la forme d'origine)
df_proj["score_prog"] = df_proj.groupby("pays")["score_prog"].transform(
    lambda s: s.fillna(s.median())
)

# 3) Score final
df_proj["score_final"] = 0.5 * df_proj["score_qcm"] + 0.5 * df_proj["score_prog"]

# 4) Mois d'inscription
df_proj["mois_inscription"] = df_proj["date_inscription"].dt.month

# 5) Pivot table
piv_final = df_proj.pivot_table(
    index="pays", columns="mois_inscription", values="score_final", aggfunc="mean"
)
print("Pivot pays x mois :\n", piv_final.round(1), "\n")

# 6) Optimisation dtypes
df_proj["pays"] = df_proj["pays"].astype("category")
df_proj["age"] = pd.to_numeric(df_proj["age"], downcast="integer")
print("dtypes optimisés :\n", df_proj.dtypes, "\n")

# 7) Préparation ML
df_proj["admis"] = np.where(df_proj["score_final"] > 70, 1, 0)
X = df_proj[["age", "score_qcm", "score_prog"]].to_numpy()
y = df_proj["admis"].to_numpy()

print("X shape :", X.shape, "| y shape :", y.shape)
print("Répartition admis/non-admis :", np.bincount(y))


Pivot pays x mois :
 mois_inscription     1
pays                  
BF                66.5
CI                65.8
NE                68.8
SN                67.2
TG                68.5 

dtypes optimisés :
 candidat_id                  int64
pays                      category
age                           int8
score_qcm                  float64
score_prog                 float64
date_inscription    datetime64[us]
score_final                float64
mois_inscription             int32
dtype: object 

X shape : (500, 3) | y shape : (500,)
Répartition admis/non-admis : [295 205]


---
# Pour aller plus loin (après ces 3 s)

- **NumPy** en profondeur (broadcasting, algèbre linéaire) prérequis direct pour comprendre les couches d'un réseau de neurones.
- **Matplotlib / Seaborn** pour visualiser les données nettoyées avec pandas.
- **scikit-learn** : `train_test_split`, `Pipeline`, `ColumnTransformer` le prolongement naturel du feature engineering pandas.
- **Polars** : une alternative pandas plus rapide (utile si les datasets IOAI sont volumineux).
- Documentation officielle pandas : https://pandas.pydata.org/docs/ (excellent index de méthodes par thème).

### Rappel des mémo-techniques clés
- **LOC = Label**, **ILOC = Integer**.
- **GroupBy = Split → Apply → Combine**.
- **MERGE = JOIN SQL**, **CONCAT = empiler**.
- **np.where = if/else vectorisé**, **np.select = switch/case vectorisé**.
- **category/downcast = optimisation mémoire**.
- **Boucle for = à pied, vectorisation = en voiture.**

Bon courage pour la préparation IOAI 2026
